In [32]:
import os
import random
import numpy as np
from PIL import Image, ImageEnhance

INPUT_DIR = "Textures/floor/seamless_tiles_32"
OUTPUT_DIR = "outputs/baseline_A/mirror"

NUM_SAMPLES = 100  # keep same for all methods

os.makedirs(OUTPUT_DIR, exist_ok=True)

def perturb(img):
    # very small brightness jitter (±5%)
    enhancer = ImageEnhance.Brightness(img)
    img = enhancer.enhance(random.uniform(0.95, 1.05))

    # tiny Gaussian noise
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, 2, arr.shape)
    arr = np.clip(arr + noise, 0, 255)

    return Image.fromarray(arr.astype(np.uint8))

tiles = [
    Image.open(os.path.join(INPUT_DIR, f)).convert("RGB")
    for f in os.listdir(INPUT_DIR)
    if f.endswith(".png")
]

assert len(tiles) > 0, "No seamless tiles found!"

for i in range(NUM_SAMPLES):
    img = random.choice(tiles).copy()

    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    if random.random() < 0.5:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)

    img = img.rotate(random.choice([0, 90, 180, 270]))

    img = perturb(img)
    img.save(f"{OUTPUT_DIR}/mirror_{i:03d}.png")

print("Baseline A1 (Mirror) complete.")


Baseline A1 (Mirror) complete.


In [34]:
import os
import random
import numpy as np
from PIL import Image

OUTPUT_DIR = "outputs/baseline_A/quilting"

NUM_SAMPLES = 100
PATCH_SIZE = 8
OUT_SIZE = 32

os.makedirs(OUTPUT_DIR, exist_ok=True)

tiles = []
for f in os.listdir(INPUT_DIR):
    if f.endswith(".png"):
        img = Image.open(os.path.join(INPUT_DIR, f)).convert("RGB")
        tiles.append(np.array(img))

assert len(tiles) > 0, "No seamless tiles found!"

def random_patch():
    img = random.choice(tiles)
    h, w, _ = img.shape
    x = random.randint(0, w - PATCH_SIZE)
    y = random.randint(0, h - PATCH_SIZE)
    return img[y:y+PATCH_SIZE, x:x+PATCH_SIZE]

for i in range(NUM_SAMPLES):
    canvas = np.zeros((OUT_SIZE, OUT_SIZE, 3), dtype=np.uint8)

    for y in range(0, OUT_SIZE, PATCH_SIZE):
        for x in range(0, OUT_SIZE, PATCH_SIZE):
            canvas[y:y+PATCH_SIZE, x:x+PATCH_SIZE] = random_patch()

    Image.fromarray(canvas).save(f"{OUTPUT_DIR}/quilt_{i:03d}.png")

print("Baseline A2 (Quilting) complete.")


Baseline A2 (Quilting) complete.


In [36]:
import os
from PIL import Image

INPUT_DIR = "outputs/baseline_A/mirror"   # or quilting
OUTPUT_PATH = "outputs/baseline_A/tiled_previews/mirror_grid.png"

TILE_FACTOR = 8

files = sorted(os.listdir(INPUT_DIR))
img = Image.open(os.path.join(INPUT_DIR, files[0]))

w, h = img.size
assert (w, h) == (32, 32), "Tile is not 32x32!"

grid = Image.new("RGB", (w * TILE_FACTOR, h * TILE_FACTOR))

for y in range(TILE_FACTOR):
    for x in range(TILE_FACTOR):
        grid.paste(img, (x * w, y * h))

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)
grid.save(OUTPUT_PATH)

print("Tiled preview saved.")


Tiled preview saved.
